# Validasi Akhir Dataset: Multiclass Language Identification
Notebook ini memuat tahapan pemuatan data, pembersihan, dan validasi 10 poin sesuai instruksi.

In [ ]:
import pandas as pd
import numpy as np
import os
import unicodedata
import random
from sklearn.model_selection import train_test_split

# Poin 10: Random seed harus ditetapkan
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
print(f"Random seed di-set ke: {RANDOM_SEED}")


In [ ]:
# Definisi Dataset Rekomendasi
datasets_info = [
    {"file": "acehnese.csv", "text_col": "acehnese", "label": "acehnese", "encoding": "ascii"},
    {"file": "balinese.csv", "text_col": "balinese", "label": "balinese", "encoding": "ascii"},
    {"file": "banjarese.csv", "text_col": "banjarese", "label": "banjarese", "encoding": "ascii"},
    {"file": "buginese.csv", "text_col": "buginese", "label": "buginese", "encoding": "ascii"},
    {"file": "madurese.csv", "text_col": "madurese", "label": "madurese", "encoding": "utf-8"},
    {"file": "minangkabau.csv", "text_col": "minangkabau", "label": "minangkabau", "encoding": "ascii"},
    {"file": "ngaju.csv", "text_col": "ngaju", "label": "ngaju", "encoding": "ascii"},
    {"file": "sundanese.csv", "text_col": "sundanese", "label": "sundanese", "encoding": "ascii"},
    {"file": "thai-government-corpus.csv", "text_col": "text", "label": "thai", "encoding": "utf-8"},
    {"file": "toba_batak.csv", "text_col": "toba_batak", "label": "toba_batak", "encoding": "ascii"}
]

dataset_dir = "datasets"
dataframes = []
berhasil = []
gagal = []

# Poin 1, 2 & 9: Pemuatan Data dan Kolom
print("--- STATUS PEMUATAN DATA ---")
for info in datasets_info:
    file_path = os.path.join(dataset_dir, info["file"])
    try:
        df_temp = pd.read_csv(file_path, encoding=info["encoding"])
        # Simpan kolom teks asli untuk log
        original_cols = list(df_temp.columns)
        
        # Standardize
        df_temp = df_temp[[info["text_col"]]].copy()
        df_temp.columns = ["text"]
        df_temp["label"] = info["label"]
        
        dataframes.append(df_temp)
        berhasil.append(info['file'])
        print(f"[SUKSES] File: {info['file']} | Kolom asli: {original_cols} -> Diekstrak: '{info['text_col']}' (Target label: {info['label']})")
    except Exception as e:
        gagal.append((info['file'], str(e)))
        print(f"[GAGAL]  File: {info['file']} | Error: {e}")

print("\n--- POIN 1: DAFTAR FILE BERHASIL ---")
for f in berhasil: print(f"- {f}")

print("\n--- POIN 9: DAFTAR FILE GAGAL ---")
if not gagal: print("Tidak ada file yang gagal dibaca.")
for f, err in gagal: print(f"- {f}: {err}")


In [ ]:
# Penggabungan
df_all = pd.concat(dataframes, ignore_index=True)

dist_awal = df_all['label'].value_counts().to_dict()
baris_awal = len(df_all)

# Pembersihan Null & Duplikat
df_all = df_all.dropna(subset=['text'])
baris_setelah_null = len(df_all)

df_all = df_all.drop_duplicates(subset=['text'])
baris_setelah_duplikat = len(df_all)

print("--- POIN 3: STATISTIK BARIS ---")
print(f"Total baris awal                : {baris_awal}")
print(f"Total baris (setelah drop null) : {baris_setelah_null} (Dihapus: {baris_awal - baris_setelah_null})")
print(f"Total baris (setelah drop dup)  : {baris_setelah_duplikat} (Dihapus: {baris_setelah_null - baris_setelah_duplikat})")


In [ ]:
def clean_text(t):
    if not isinstance(t, str):
        return ""
    t = unicodedata.normalize('NFKC', t)
    t = t.strip()
    return t

df_all['text'] = df_all['text'].apply(clean_text)

# Poin 8: Pemeriksaan teks kosong atau terlalu pendek (< 3 char)
terlalu_pendek = df_all[df_all['text'].str.len() < 3]
print("--- POIN 8: TEKS TERLALU PENDEK (< 3 KARAKTER) ---")
print(f"Ditemukan {len(terlalu_pendek)} baris teks yang terlalu pendek.")
if len(terlalu_pendek) > 0:
    print("Contoh data yang dihapus:")
    print(terlalu_pendek.head())

# Buang teks terlalu pendek dan deduplikasi setelah normalisasi agar split bebas overlap.
df_all = df_all[df_all['text'].str.len() >= 3].drop_duplicates(subset=['text']).reset_index(drop=True)
dist_akhir = df_all['label'].value_counts().to_dict()
print(f"\nBaris setelah pembersihan akhir: {len(df_all)}")

In [ ]:
print("--- POIN 4: DISTRIBUSI JUMLAH TEKS SEBELUM & SESUDAH ---")
dist_akhir = df_all['label'].value_counts().to_dict()

print(f"{'Bahasa':<15} | {'Sebelum':<10} | {'Sesudah':<10} | {'Selisih (Dibuang)'}")
print("-"*60)
for lang in dist_awal.keys():
    awal = dist_awal.get(lang, 0)
    akhir = dist_akhir.get(lang, 0)
    print(f"{lang:<15} | {awal:<10} | {akhir:<10} | {awal - akhir}")


In [ ]:
print("--- POIN 5: LIMA CONTOH TEKS DARI SETIAP BAHASA ---")
for lang in df_all['label'].unique():
    print(f"\n>>> Bahasa: {lang}")
    sampel = df_all[df_all['label'] == lang]['text'].sample(n=min(5, dist_akhir[lang]), random_state=RANDOM_SEED)
    for i, teks in enumerate(sampel):
        print(f"  {i+1}. {teks[:100]}{'...' if len(teks)>100 else ''}")


In [ ]:
# Pembagian Data (70/15/15)
# Guard terakhir: satu teks hanya boleh berada pada satu baris sebelum stratified split.
df_all = df_all.copy()
df_all['text'] = df_all['text'].map(clean_text)
df_all = df_all[df_all['text'].str.len() >= 3].drop_duplicates(subset=['text']).reset_index(drop=True)
dist_akhir = df_all['label'].value_counts().to_dict()
assert not df_all['text'].duplicated().any(), "Teks duplikat masih tersisa sebelum split"

df_temp, df_test = train_test_split(
    df_all, test_size=0.15, random_state=RANDOM_SEED, stratify=df_all['label']
)
df_train, df_val = train_test_split(
    df_temp, test_size=(0.15/0.85), random_state=RANDOM_SEED, stratify=df_temp['label']
)

print("--- POIN 6: JUMLAH DATA TRAIN, VAL, TEST ---")
train_dist = df_train['label'].value_counts()
val_dist = df_val['label'].value_counts()
test_dist = df_test['label'].value_counts()

print(f"{'Bahasa':<15} | {'Train':<8} | {'Val':<8} | {'Test':<8}")
print("-"*50)
for lang in dist_akhir.keys():
    print(f"{lang:<15} | {train_dist.get(lang, 0):<8} | {val_dist.get(lang, 0):<8} | {test_dist.get(lang, 0):<8}")

In [ ]:
print("--- POIN 7: PEMERIKSAAN KEBOCORAN TEKS (DATA LEAKAGE) ---")
set_train = set(df_train['text'])
set_val = set(df_val['text'])
set_test = set(df_test['text'])

bocor_train_val = set_train.intersection(set_val)
bocor_train_test = set_train.intersection(set_test)
bocor_val_test = set_val.intersection(set_test)

if bocor_train_val or bocor_train_test or bocor_val_test:
    print("[PERINGATAN] Ada kebocoran data antar partisi!")
    print(f"Train & Val overlap: {len(bocor_train_val)} baris")
    print(f"Train & Test overlap: {len(bocor_train_test)} baris")
    print(f"Val & Test overlap: {len(bocor_val_test)} baris")
else:
    print("[AMAN] Tidak ada teks yang tumpang tindih antara Train, Validation, dan Test.")


## 6. Model Baseline (TF-IDF Karakter + Logistic Regression)
Pipeline dilatih hanya pada train set. Validation digunakan untuk memilih baseline, sedangkan test hanya disentuh sekali pada evaluasi final.

In [ ]:
import json
from itertools import combinations
from pathlib import Path

import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# Konfigurasi baseline ditetapkan sebelum fit; vectorizer hanya melihat train set.
PREPROCESSING_CONFIG = {
    "analyzer": "char_wb",
    "ngram_range": [2, 5],
    "lowercase": True,
    "min_df": 2,
    "sublinear_tf": True,
    "max_features": 100000,
}
MODEL_CONFIG = {
    "max_iter": 1000,
    "class_weight": "balanced",
    "random_state": RANDOM_SEED,
}
CONFIDENCE_THRESHOLD = 0.50
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

labels = sorted(df_all["label"].unique())
X_train, y_train = df_train["text"].astype(str), df_train["label"]
X_val, y_val = df_val["text"].astype(str), df_val["label"]
X_test, y_test = df_test["text"].astype(str), df_test["label"]

vectorizer = TfidfVectorizer(
    analyzer=PREPROCESSING_CONFIG["analyzer"],
    ngram_range=tuple(PREPROCESSING_CONFIG["ngram_range"]),
    lowercase=PREPROCESSING_CONFIG["lowercase"],
    min_df=PREPROCESSING_CONFIG["min_df"],
    sublinear_tf=PREPROCESSING_CONFIG["sublinear_tf"],
    max_features=PREPROCESSING_CONFIG["max_features"],
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(**MODEL_CONFIG)
model.fit(X_train_tfidf, y_train)


def evaluate_split(split_name, features, targets):
    predictions = model.predict(features)
    metrics = {
        "accuracy": float(accuracy_score(targets, predictions)),
        "macro_precision": float(precision_score(targets, predictions, labels=labels, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(targets, predictions, labels=labels, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(targets, predictions, labels=labels, average="macro", zero_division=0)),
    }
    print(f"\n--- HASIL {split_name.upper()} ---")
    print(f"Accuracy        : {metrics['accuracy']:.4f}")
    print(f"Macro Precision : {metrics['macro_precision']:.4f}")
    print(f"Macro Recall    : {metrics['macro_recall']:.4f}")
    print(f"Macro F1-score  : {metrics['macro_f1']:.4f}")
    print("\nClassification report:")
    print(classification_report(targets, predictions, labels=labels, zero_division=0))
    matrix = confusion_matrix(targets, predictions, labels=labels)
    print("Confusion matrix (baris=aktual, kolom=prediksi):")
    print(pd.DataFrame(matrix, index=labels, columns=labels))
    return metrics, predictions, matrix


# Validation adalah dasar pemilihan baseline; test belum digunakan sampai tahap final.
validation_metrics, val_predictions, val_matrix = evaluate_split("validation", X_val_tfidf, y_val)
selected_parameters = {"preprocessing": PREPROCESSING_CONFIG, "model": MODEL_CONFIG}
print("\nParameter baseline dipilih berdasarkan hasil validation di atas.")

# Evaluasi final dilakukan tepat setelah model baseline dipilih.
test_metrics, test_predictions, test_matrix = evaluate_split("test final", X_test_tfidf, y_test)

# Bahasa yang paling sering tertukar, tidak menghitung prediksi benar.
confusions = []
for actual, predicted in combinations(labels, 2):
    count = int(test_matrix[labels.index(actual), labels.index(predicted)] + test_matrix[labels.index(predicted), labels.index(actual)])
    if count:
        confusions.append((count, actual, predicted))
print("\nBahasa yang paling sering tertukar (test):")
for count, first, second in sorted(confusions, reverse=True)[:10]:
    print(f"- {first} <-> {second}: {count}")

wrong_mask = y_test.to_numpy() != test_predictions
wrong_examples = pd.DataFrame({
    "text": X_test.to_numpy()[wrong_mask],
    "actual": y_test.to_numpy()[wrong_mask],
    "predicted": test_predictions[wrong_mask],
}).head(10)
print("\nContoh prediksi yang salah (test):")
print(wrong_examples.to_string(index=False) if len(wrong_examples) else "Tidak ada prediksi yang salah.")


def predict_language(text):
    """Prediksi bahasa, confidence, dan tiga kandidat teratas."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError("text harus berupa string yang tidak kosong")
    probabilities = model.predict_proba(vectorizer.transform([text]))[0]
    ranked = np.argsort(probabilities)[::-1][:3]
    top_candidates = [
        {"language": str(model.classes_[index]), "confidence": float(probabilities[index])}
        for index in ranked
    ]
    prediction = top_candidates[0]["language"]
    confidence = top_candidates[0]["confidence"]
    if confidence < CONFIDENCE_THRESHOLD:
        prediction = "bahasa tidak dikenali"
    return {
        "predicted_language": prediction,
        "confidence": confidence,
        "top_3_candidates": top_candidates,
    }

print("\n--- UJI predict_language (teks buatan) ---")
synthetic_texts = {
    "indonesian_like": "Saya sedang membaca berita dan belajar bahasa hari ini.",
    "thai_like": "à¸§à¸±à¸™à¸™à¸µà¹‰à¸‰à¸±à¸™à¸à¸³à¸¥à¸±à¸‡à¸­à¹ˆà¸²à¸™à¸‚à¹ˆà¸²à¸§à¹à¸¥à¸°à¹€à¸£à¸µà¸¢à¸™à¸ à¸²à¸©à¸²",
    "sundanese_like": "Abdi bade maca berita sareng diajar basa ayeuna.",
    "buginese_like": "Iya manengka lao makkita bicara.",
}
prediction_examples = []
for name, text in synthetic_texts.items():
    result = predict_language(text)
    prediction_examples.append({"example": name, **result})
    print(f"{name}: {result}")
print("Catatan: ambang 0,50 adalah ambang awal dan perlu dikalibrasi menggunakan validation set.")

metrics_evaluation = {
    "validation": validation_metrics,
    "test": test_metrics,
    "labels": labels,
    "n_languages": len(labels),
    "n_train": len(df_train),
    "n_validation": len(df_val),
    "n_test": len(df_test),
}
joblib.dump(
    {"vectorizer": vectorizer, "model": model},
    ARTIFACT_DIR / "language_detector_baseline.joblib",
)
with open(ARTIFACT_DIR / "labels.json", "w", encoding="utf-8") as file:
    json.dump(labels, file, ensure_ascii=False, indent=2)
with open(ARTIFACT_DIR / "preprocessing_config.json", "w", encoding="utf-8") as file:
    json.dump({"vectorizer": PREPROCESSING_CONFIG, "model": MODEL_CONFIG}, file, indent=2)
with open(ARTIFACT_DIR / "metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics_evaluation, file, indent=2)
with open(ARTIFACT_DIR / "confidence_threshold.json", "w", encoding="utf-8") as file:
    json.dump({"threshold": CONFIDENCE_THRESHOLD, "note": "Ambang awal; kalibrasikan dengan validation set."}, file, indent=2)
pd.DataFrame(test_matrix, index=labels, columns=labels).to_csv(ARTIFACT_DIR / "test_confusion_matrix.csv")
wrong_examples.to_csv(ARTIFACT_DIR / "test_misclassified_examples.csv", index=False)

model_path = ARTIFACT_DIR / "language_detector_baseline.joblib"
print("\n=== RINGKASAN BASELINE ===")
print(f"Jumlah bahasa       : {len(labels)}")
print(f"Jumlah train/val/test: {len(df_train)}/{len(df_val)}/{len(df_test)}")
print(f"Accuracy test       : {test_metrics['accuracy']:.4f}")
print(f"Macro F1-score test : {test_metrics['macro_f1']:.4f}")
print(f"Ukuran file model   : {model_path.stat().st_size / (1024 ** 2):.2f} MB")
print(f"Lokasi artifact     : {ARTIFACT_DIR.resolve()}")
print("Contoh hasil prediksi:")
for example in prediction_examples:
    print(f"- {example['example']}: {example['predicted_language']} (confidence={example['confidence']:.4f})")

## 7. Diagnosis dan Perbaikan Baseline
Bagian ini hanya mendiagnosis baseline yang sudah dievaluasi. Test set digunakan sebagai catatan hasil baseline, bukan untuk tuning atau pemilihan model.

In [ ]:
import re
from unicodedata import normalize

print("=== DIAGNOSIS BASELINE ===")
print("Test set hanya digunakan untuk mencatat diagnosis baseline; tidak ada tuning dilakukan.")

# 1. Jumlah data per bahasa dan partisi.
partition_counts = pd.DataFrame({
    "train": y_train.value_counts(),
    "validation": y_val.value_counts(),
    "test": y_test.value_counts(),
}).fillna(0).astype(int).reindex(labels).fillna(0).astype(int)
print("\n--- Jumlah data per bahasa ---")
print(partition_counts.to_string())

# 2. Metrik per bahasa, diurutkan dari F1 terendah.
report = classification_report(y_test, test_predictions, labels=labels, output_dict=True, zero_division=0)
per_language_metrics = pd.DataFrame([
    {"language": language, "precision": report[language]["precision"], "recall": report[language]["recall"], "f1_score": report[language]["f1-score"]}
    for language in labels
]).sort_values(["f1_score", "recall", "language"])
print("\n--- Precision, recall, F1 per bahasa (terendah lebih dulu) ---")
print(per_language_metrics.to_string(index=False, formatters={column: "{:.4f}".format for column in ["precision", "recall", "f1_score"]}))

# 3. Confusion matrix dengan label eksplisit.
print("\n--- Confusion matrix test (aktual x prediksi) ---")
confusion_table = pd.DataFrame(test_matrix, index=labels, columns=labels)
print(confusion_table.to_string())

# 4-5. Lima belas pasangan tertukar dan maksimal 10 contoh per pasangan.
pair_records = []
for actual_index, actual in enumerate(labels):
    for predicted_index, predicted in enumerate(labels):
        if actual != predicted and test_matrix[actual_index, predicted_index] > 0:
            pair_records.append({
                "count": int(test_matrix[actual_index, predicted_index]),
                "actual": actual,
                "predicted": predicted,
            })
top_pairs = pd.DataFrame(pair_records).sort_values(["count", "actual", "predicted"], ascending=[False, True, True]).head(15)
print("\n--- 15 pasangan bahasa paling sering tertukar ---")
print(top_pairs.to_string(index=False) if len(top_pairs) else "Tidak ada kesalahan klasifikasi.")

misclassified_records = []
for pair in top_pairs.itertuples(index=False):
    pair_mask = (y_test.to_numpy() == pair.actual) & (test_predictions == pair.predicted)
    pair_indices = np.flatnonzero(pair_mask)[:10]
    print(f"\n{pair.actual} -> {pair.predicted} ({pair.count} kasus; menampilkan maksimal 10):")
    for index in pair_indices:
        text = X_test.iloc[index]
        print(f"- {text[:300]}")
        misclassified_records.append({"actual": pair.actual, "predicted": pair.predicted, "text": text})
misclassified_examples = pd.DataFrame(misclassified_records, columns=["actual", "predicted", "text"])

# 6. Statistik panjang karakter dan kata per bahasa.
def word_count(text):
    return len(re.findall(r"\\S+", text))

length_rows = []
for language, group in df_all.groupby("label"):
    character_lengths = group["text"].astype(str).str.len()
    word_lengths = group["text"].astype(str).map(word_count)
    for unit, values in [("characters", character_lengths), ("words", word_lengths)]:
        length_rows.append({
            "language": language,
            "measure": unit,
            "minimum": int(values.min()),
            "median": float(values.median()),
            "mean": float(values.mean()),
            "maximum": int(values.max()),
        })
length_stats = pd.DataFrame(length_rows)
print("\n--- Statistik panjang teks ---")
print(length_stats.to_string(index=False))

# 7. Audit kualitas teks pada data kerja dan data mentah yang dibaca ulang secara read-only.
def normalized_text(text):
    text = normalize("NFKC", str(text)).lower().strip()
    return re.sub(r"\\s+", " ", text)

def quality_flags(text):
    value = str(text)
    stripped = value.strip()
    return {
        "empty": not bool(stripped),
        "too_short_lt_3": len(stripped) < 3,
        "only_numbers_symbols": bool(stripped) and not bool(re.search(r"[A-Za-z\\u0E00-\\u0E7F]", stripped)),
        "broken_encoding": bool(re.search(r"ï¿½|Ãƒ.|Ã‚.|Ã¢[â‚¬â„¢â‚¬Å“â‚¬Â¢]", value)),
        "metadata": bool(re.search(r"(?:^|[|;])\\s*(?:id|url|title|author|date|label|source)\\s*[:=]", value, re.I)),
        "html": bool(re.search(r"<[^>]+>", value)),
        "url": bool(re.search(r"(?:https?://|www\\.)\\S+", value, re.I)),
    }

quality_frame = df_all[["text", "label"]].copy()
quality_frame["normalized_text"] = quality_frame["text"].map(normalized_text)
quality_flags_frame = pd.DataFrame(quality_frame["text"].map(quality_flags).tolist(), index=quality_frame.index)
quality_counts = quality_flags_frame.sum().sort_values(ascending=False)
print("\n--- Temuan kualitas teks pada data kerja ---")
print(quality_counts.to_string())

raw_records = []
for info in datasets_info:
    raw_path = os.path.join(dataset_dir, info["file"])
    raw_df = pd.read_csv(raw_path, encoding=info["encoding"])
    for _, row in raw_df.iterrows():
        raw_records.append({
            "dataset": info["file"],
            "label": info["label"],
            "text": row.get(info["text_col"], ""),
            "indonesian_reference": row.get("indonesian", ""),
            "english_reference": row.get("english", ""),
        })
raw_frame = pd.DataFrame(raw_records).fillna("")
raw_frame["normalized_text"] = raw_frame["text"].map(normalized_text)
raw_quality = pd.DataFrame(raw_frame["text"].map(quality_flags).tolist())
print("\n--- Temuan kualitas teks pada data mentah (audit read-only) ---")
print(raw_quality.sum().sort_values(ascending=False).to_string())

# 8. Duplicate exact/normalized dengan label berbeda, sebelum deduplikasi pipeline.
exact_cross_label = raw_frame[raw_frame["text"].astype(str).duplicated(keep=False)].copy()
exact_cross_label = exact_cross_label.groupby("text").filter(lambda group: group["label"].nunique() > 1)
normalized_cross_label = raw_frame[raw_frame["normalized_text"].duplicated(keep=False)].copy()
normalized_cross_label = normalized_cross_label.groupby("normalized_text").filter(lambda group: group["label"].nunique() > 1)
print(f"\nExact duplicate lintas label       : {exact_cross_label['text'].nunique()}")
print(f"Normalized duplicate lintas label : {normalized_cross_label['normalized_text'].nunique()}")

# 9. Indikasi campuran bahasa atau terjemahan Indonesia/Inggris per dataset.
indonesian_markers = re.compile(r"\\b(dan|yang|dengan|untuk|dari|ini|itu|tidak|adalah|saya|kami)\\b", re.I)
english_markers = re.compile(r"\\b(the|and|with|for|from|this|that|not|is|are|of|to)\\b", re.I)
source_rows = []
for dataset, group in raw_frame.groupby("dataset"):
    target_text = group["text"].astype(str)
    indonesian_reference = group["indonesian_reference"].astype(str)
    exact_translation = (target_text.map(normalized_text) == indonesian_reference.map(normalized_text)).mean() if indonesian_reference.ne("").any() else 0.0
    indonesian_hits = target_text.map(lambda value: bool(indonesian_markers.search(value))).mean()
    english_hits = target_text.map(lambda value: bool(english_markers.search(value))).mean()
    source_rows.append({
        "dataset": dataset,
        "rows": len(group),
        "indonesian_translation_like_fraction": float(exact_translation),
        "indonesian_marker_fraction": float(indonesian_hits),
        "english_marker_fraction": float(english_hits),
        "possible_mixed_language": bool(indonesian_hits > 0.20 and english_hits > 0.20),
        "possible_non_target_text": bool(indonesian_hits > 0.35 or english_hits > 0.35),
    })
source_audit = pd.DataFrame(source_rows).sort_values("possible_non_target_text", ascending=False)
print("\n--- Audit campuran bahasa/terjemahan per dataset ---")
print(source_audit.to_string(index=False))

# 10. Distribusi confidence benar vs salah; threshold 0,50 tidak dioptimalkan.
test_probabilities = model.predict_proba(X_test_tfidf)
test_confidences = test_probabilities.max(axis=1)
confidence_frame = pd.DataFrame({
    "correct": y_test.to_numpy() == test_predictions,
    "confidence": test_confidences,
})
confidence_summary = confidence_frame.groupby("correct")["confidence"].agg(["count", "min", "median", "mean", "max"])
confidence_summary.index = ["wrong" if not value else "correct" for value in confidence_summary.index]
confidence_bins = [0.0, 0.25, 0.50, 0.75, 1.0]
confidence_distribution = pd.crosstab(
    pd.cut(confidence_frame["confidence"], bins=confidence_bins, include_lowest=True),
    confidence_frame["correct"],
).rename(columns={True: "correct", False: "wrong"})
print("\n--- Distribusi confidence ---")
print(confidence_summary.to_string())
print(confidence_distribution.to_string())
print("Threshold 0,50 tidak dianggap optimal dan belum dituning menggunakan test set.")

# Rekomendasi konkret berdasarkan F1 dan audit kualitas, tanpa retraining.
recommendations = []
for row in per_language_metrics.itertuples(index=False):
    count = int(partition_counts.loc[row.language, "train"])
    if row.f1_score < 0.50 or row.recall < 0.50:
        action = "Tambah data berlabel dan audit pasangan bahasa yang paling sering tertukar."
        if count < 500:
            action = "Prioritas tinggi: tambah data berlabel; jumlah train relatif kecil."
        recommendations.append({"language": row.language, "priority": "high", "recommendation": action})
for row in source_audit.itertuples(index=False):
    if row.possible_non_target_text or row.possible_mixed_language:
        recommendations.append({"language": row.dataset, "priority": "high", "recommendation": "Audit isi dataset, pisahkan campuran bahasa/terjemahan dan bersihkan metadata atau teks non-target."})
recommendation_frame = pd.DataFrame(recommendations, columns=["language", "priority", "recommendation"])
print("\n--- Rekomendasi ---")
print(recommendation_frame.to_string(index=False) if len(recommendation_frame) else "Tidak ada rekomendasi otomatis.")

# Satu CSV ringkas dengan kategori diagnosis dan nilai utama.
diagnosis_rows = []
for row in partition_counts.reset_index(names="language").to_dict("records"):
    for split in ["train", "validation", "test"]:
        diagnosis_rows.append({"category": "partition_count", "item": row["language"], "metric": split, "value": row[split]})
for row in per_language_metrics.to_dict("records"):
    for metric in ["precision", "recall", "f1_score"]:
        diagnosis_rows.append({"category": "per_language_metric", "item": row["language"], "metric": metric, "value": row[metric]})
for metric, value in quality_counts.items():
    diagnosis_rows.append({"category": "quality_flag", "item": "all_data", "metric": metric, "value": int(value)})
for metric, value in confidence_summary.stack().items():
    diagnosis_rows.append({"category": "confidence_summary", "item": metric[0], "metric": metric[1], "value": float(value)})
for row in recommendation_frame.to_dict("records"):
    diagnosis_rows.append({"category": "recommendation", "item": row["language"], "metric": row["priority"], "value": row["recommendation"]})

diagnosis_path = ARTIFACT_DIR / "baseline_diagnosis.csv"
misclassified_path = ARTIFACT_DIR / "misclassified_examples.csv"
pd.DataFrame(diagnosis_rows).to_csv(diagnosis_path, index=False)
misclassified_examples.to_csv(misclassified_path, index=False)
print(f"\nDiagnosis tersimpan: {diagnosis_path.resolve()}")
print(f"Kesalahan tersimpan: {misclassified_path.resolve()}")
print("Diagnosis selesai; tidak ada pelatihan ulang dilakukan.")